In [ ]:
using Plots, Polynomials, LaTeXStrings, ColorSchemes, DelimitedFiles, DataFrames, Interpolations
using Statistics, StatsPlots, Random, ProgressMeter, Printf, LinearAlgebra, Plots.PlotMeasures
include("plot_classification.jl")
include("plot_power_energy.jl")
include("utils.jl");

In [ ]:
# Modifying backend GR attributes
default(fmt = :png);

# SW-HW agreement computation

In [ ]:
n = 66 # or 66
include("signals_$(n).jl") # SW signals

dt = 0.01  # seconds
time_SW = collect(0:100) .* dt;  # t[1] = 0.0, t[101] = 1.0

In [ ]:
# Loading HW signals
time_HW = readdlm("./Cadence_output/agreement_time_HW_$(n).dat")

logits = readdlm("./Cadence_output/agreement_logits_$(n).dat")

result_after_skip = readdlm("./Cadence_output/agreement_result_after_skip_$(n).dat")

Iin_cell_data1_1 = readdlm("./Cadence_output/Iin_cell_data1_1_$(n).dat")
Iout_cell_data1_1 = readdlm("./Cadence_output/Iout_cell_data1_1_$(n).dat")
Iin_cell_data1_2 = readdlm("./Cadence_output/Iin_cell_data1_2_$(n).dat")
Iout_cell_data1_2 = readdlm("./Cadence_output/Iout_cell_data1_2_$(n).dat")
Iin_cell_data1_3 = readdlm("./Cadence_output/Iin_cell_data1_3_$(n).dat")
Iout_cell_data1_3 = readdlm("./Cadence_output/Iout_cell_data1_3_$(n).dat")
Iin_cell_data1_4 = readdlm("./Cadence_output/Iin_cell_data1_4_$(n).dat")
Iout_cell_data1_4 = readdlm("./Cadence_output/Iout_cell_data1_4_$(n).dat")

Iin_cell_data2_1 = readdlm("./Cadence_output/Iin_cell_data2_1_$(n).dat")
Iout_cell_data2_1 = readdlm("./Cadence_output/Iout_cell_data2_1_$(n).dat")
Iin_cell_data2_2 = readdlm("./Cadence_output/Iin_cell_data2_2_$(n).dat")
Iout_cell_data2_2 = readdlm("./Cadence_output/Iout_cell_data2_2_$(n).dat")
Iin_cell_data2_3 = readdlm("./Cadence_output/Iin_cell_data2_3_$(n).dat")
Iout_cell_data2_3 = readdlm("./Cadence_output/Iout_cell_data2_3_$(n).dat")
Iin_cell_data2_4 = readdlm("./Cadence_output/Iin_cell_data2_4_$(n).dat")
Iout_cell_data2_4 = readdlm("./Cadence_output/Iout_cell_data2_4_$(n).dat");

In [ ]:
# ============================================================
# HW vs SW inference comparison — all signals
# ============================================================
# Assumes in scope:
#   - inference_vectors_v2.jl  (time_SW, all L1/L2 vectors, logit_*, pred)
#   - Cadence loading block    (time_HW, Iin/Iout_cell_data{1,2}_{1..4},
#                               out, s_pos, s_neg, result_after_skip, logits)
# ============================================================

using Plots
gr()

const COL_HW  = :steelblue
const COL_SW  = :crimson
const LW_HW   = 1.2
const FIGSIZE = (800, 800)
const DPI     = 150

# ---- manual step expansion (avoids GR steppre bug) ----------
function make_step(t, y)
    t2 = vec(vcat(t[1:end-1]', t[2:end]'))
    y2 = vec(vcat(y[1:end-1]', y[1:end-1]'))
    push!(t2, t[end]); push!(y2, y[end])
    return t2, y2
end

# ---- one overlay panel: HW continuous / SW staircase --------
function comp_panel(tHW, hw, tSW, sw, ttl;
                    lbl_hw=L"\mathrm{HW}\,\,\mathrm{(nA)}", lbl_sw=L"\mathrm{SW}\,\,\mathrm{(a.u.)}",
                    xlabel_text="", xticks_label=["", "", "", "", "", ""])
    tSW_s, sw_s = make_step(tSW, sw)
    p = plot(tHW, hw;
             label=lbl_hw, color=COL_HW, lw=LW_HW,
             title="", titlefontsize=9,
             xlabel=xlabel_text, ylabel=L"\mathrm{Current/signal}",
             xticks=([0, 0.2, 0.4, 0.6, 0.8, 1], xticks_label),
             legend=:topright, legendfontsize=7)
    plot!(p, tSW_s, sw_s; label=lbl_sw, color=COL_SW, lw=1.2, ls=:dash)
    if maximum(hw) < 0.5 && maximum(sw) < 0.5
        ylims!((0, 0.5))
    end
    return p
end

# ---- 2×2 layout helper --------------------------------------
function fig2x2(panels, suptitle)
    plot(panels...; layout=(4,1),
         size=FIGSIZE, dpi=DPI, left_margin=5Plots.mm,
        right_margin=3Plots.mm,
        top_margin=2Plots.mm,
        bottom_margin=3Plots.mm)
end


# ============================================================
# FIGURE 1 — Layer 1 : Candidates  (Iin SBC = pre-Schmitt)
# ============================================================
fig1 = fig2x2([
    comp_panel(time_HW, Iin_cell_data1_1, time_SW, L1_cand_1, "L1  cand[1]"),
    comp_panel(time_HW, Iin_cell_data1_2, time_SW, L1_cand_2, "L1  cand[2]"),
    comp_panel(time_HW, Iin_cell_data1_3, time_SW, L1_cand_3, "L1  cand[3]"),
    comp_panel(time_HW, Iin_cell_data1_4, time_SW, L1_cand_4, "L1  cand[4]"; 
            xlabel_text=L"\mathrm{Time}\,\,\mathrm{(s)}", xticks_label=[0, 0.2, 0.4, 0.6, 0.8, 1]),
], "Layer 1 — Candidates  (pre-BMRU,  Iin BMRU)")
savefig(fig1, "./Cadence_plots/$(n)_L1_candidates.pdf")

# ============================================================
# FIGURE 2 — Layer 1 : Scaled States  (Iout SBC = binary x alpha)
# ============================================================
fig2 = fig2x2([
    comp_panel(time_HW, Iout_cell_data1_1, time_SW, L1_state_1, "L1  state[1]"),
    comp_panel(time_HW, Iout_cell_data1_2, time_SW, L1_state_2, "L1  state[2]"),
    comp_panel(time_HW, Iout_cell_data1_3, time_SW, L1_state_3, "L1  state[3]"),
    comp_panel(time_HW, Iout_cell_data1_4, time_SW, L1_state_4, "L1  state[4]"; 
            xlabel_text=L"\mathrm{Time}\,\,\mathrm{(s)}", xticks_label=[0, 0.2, 0.4, 0.6, 0.8, 1]),
], "Layer 1 — States  (Iout BMRU)")
savefig(fig2, "./Cadence_plots/$(n)_L1_states.pdf")

# ============================================================
# FIGURE 3 — Layer 2 : Candidates  (Iin SBC = pre-Schmitt)
# ============================================================
fig3 = fig2x2([
    comp_panel(time_HW, Iin_cell_data2_1, time_SW, L2_cand_1, "L2  cand[1]"),
    comp_panel(time_HW, Iin_cell_data2_2, time_SW, L2_cand_2, "L2  cand[2]"),
    comp_panel(time_HW, Iin_cell_data2_3, time_SW, L2_cand_3, "L2  cand[3]"),
    comp_panel(time_HW, Iin_cell_data2_4, time_SW, L2_cand_4, "L2  cand[4]"; 
            xlabel_text=L"\mathrm{Time}\,\,\mathrm{(s)}", xticks_label=[0, 0.2, 0.4, 0.6, 0.8, 1]),
], "Layer 2 — Candidates  (pre-BMRU,  Iin BMRU)")
savefig(fig3, "./Cadence_plots/$(n)_L2_candidates.pdf")

# ============================================================
# FIGURE 4 — Layer 2 : Scaled States  (Iout SBC = binary x alpha)
# ============================================================
fig4 = fig2x2([
    comp_panel(time_HW, Iout_cell_data2_1, time_SW, L2_state_1, "L2  state[1]"),
    comp_panel(time_HW, Iout_cell_data2_2, time_SW, L2_state_2, "L2  state[2]"),
    comp_panel(time_HW, Iout_cell_data2_3, time_SW, L2_state_3, "L2  state[3]"),
    comp_panel(time_HW, Iout_cell_data2_4, time_SW, L2_state_4, "L2  state[4]"; 
            xlabel_text=L"\mathrm{Time}\,\,\mathrm{(s)}", xticks_label=[0, 0.2, 0.4, 0.6, 0.8, 1]),
], "Layer 2 — States  (Iout BMRU)")
savefig(fig4, "./Cadence_plots/$(n)_L2_states.pdf")

# ============================================================
# FIGURE 5 — Layer 2 : Result after skip  (out + s_pos - s_neg)
# ============================================================
fig5 = fig2x2([
    comp_panel(time_HW, result_after_skip[:,1], time_SW, L2_result_1, "L2  result[1]"),
    comp_panel(time_HW, result_after_skip[:,2], time_SW, L2_result_2, "L2  result[2]"),
    comp_panel(time_HW, result_after_skip[:,3], time_SW, L2_result_3, "L2  result[3]"),
    comp_panel(time_HW, result_after_skip[:,4], time_SW, L2_result_4, "L2  result[4]"; 
            xlabel_text=L"\mathrm{Time}\,\,\mathrm{(s)}", xticks_label=[0, 0.2, 0.4, 0.6, 0.8, 1]),
], "Layer 2 — Result after skip  (Iout + Iskip)")
savefig(fig5, "./Cadence_plots/$(n)_L2_result_after_skip.pdf")

# ============================================================
# FIGURE 6 — Final logits  (w_class x result')
# ============================================================
p_c0 = comp_panel(time_HW, logits[1,:], time_SW, logit_c0, "Logit  class 0")
p_c1 = comp_panel(time_HW, logits[2,:], time_SW, logit_c1, "Logit  class 1"; 
            xlabel_text=L"\mathrm{Time}\,\,\mathrm{(s)}", xticks_label=[0, 0.2, 0.4, 0.6, 0.8, 1])

fig6 = plot(p_c0, p_c1;
            layout=(2,1),
            plot_title="",
            plot_titlefontsize=11,
            size=(FIGSIZE[1], FIGSIZE[2]÷2), dpi=DPI, left_margin=5Plots.mm,
        right_margin=3Plots.mm,
        top_margin=2Plots.mm,
        bottom_margin=3Plots.mm)
savefig(fig6, "./Cadence_plots/$(n)_logits.pdf")